---
title: "Staggered vs collocated: grid convergence of Stokes drag"
subtitle: "The two peclet.flow velocity placements on the Zick–Homsy sphere array — one grid converges onto the benchmark, the other to a value ~1% off."
author: "Peclet"
date: "2026-07-04"
categories: [flow, accuracy, staggered, collocated, Stokes, verification]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/benchmarks/staggered-vs-collocated/index.ipynb){target="_blank"}
&nbsp;Runs on a free Colab CPU runtime — the first cell installs `peclet` from PyPI.

## What this measures

`peclet.flow` ships **two velocity placements** behind one identical Python API:

- `flow.Solver` — the **staggered** MAC grid (velocities on cell faces, pressure at
  centres). The default, and *the* flow solver: exact discrete projection.
- `flow.SolverColocated` — a **collocated** grid (velocities and pressure both at
  cell centres) closed with an ABC *approximate* projection.

They solve the same equations on the same geometry; they differ only in *where the
velocity unknowns live*. This page puts that choice under a microscope on a problem
with an external ground truth — creeping flow through a periodic simple-cubic sphere
array, whose Stokes drag **Zick & Homsy (1982)** computed semi-analytically. We
refine the mesh and ask, for each grid:

1. **Does it converge, and at what order?**
2. **To *what value* does it converge** — the benchmark, or something offset from it?
3. **What does it cost** (iterations, wall-clock) to get there?

The punchline is (2): both grids are second-order *solvers*, but they do **not**
converge to the same number.

## The drag factor

A body force $F$ drives Stokes flow (no inertia) through a periodic cell of side $L$
containing one sphere of radius $R$ at solid fraction $\phi=\tfrac{4}{3}\pi R^3/L^3$.
The mean (superficial) velocity $\langle u\rangle$ fixes the dimensionless **drag
factor** — the per-sphere drag normalised by the isolated Stokes drag
$6\pi\mu R\langle u\rangle$:

$$
K=\frac{F\,L^{3}}{6\pi\mu R\,\langle u\rangle},
$$ {#eq-K}

which Zick & Homsy tabulate versus $\phi$. At $\phi=0.125$ (radius a quarter of the
cell) their value is $K_\text{ZH}=4.292$ — the target both grids must hit.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (installs peclet from PyPI on Colab/Binder)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    sys.path.insert(0, _local)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet"], check=True)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from peclet import flow

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
BLUE, RED = "#1f77b4", "#d62728"

PHI0  = 0.125                    # solid fraction (sphere radius = L/4)
K_ZH  = 4.292                    # Zick & Homsy (1982), simple cubic, at PHI0
CENTRE = [(0.5, 0.5, 0.5)]       # one sphere per period, cell-centred

## The characterization harness

The same driver runs both grids — the *only* thing that changes between the two
sweeps is the solver class passed in. It builds the sphere as a signed distance
field, drives Stokes flow to a steady state, and reports the drag factor.

Because the collocated grid's *approximate* projection leaves the mean velocity with
a small residual jitter (the staggered grid's *exact* projection does not), we detect
a moderate steady state and then report $\langle u\rangle$ **averaged over a tail of
steps** — a fair, reproducible number for both grids rather than a single jittering
sample.

In [ ]:
#| label: harness
def sphere_sdf(N, phi, centres):
    """SDF of a periodic sphere lattice on an N³ grid (negative inside a sphere)."""
    n = len(centres)
    R = (3 * phi / (4 * np.pi * n)) ** (1 / 3) * N
    g = np.arange(N) + 0.5
    X, Y, Z = np.meshgrid(g, g, g, indexing="ij")
    best = np.full((N, N, N), 1e30)
    for cx, cy, cz in centres:
        dx = X - cx * N; dx -= N * np.round(dx / N)
        dy = Y - cy * N; dy -= N * np.round(dy / N)
        dz = Z - cz * N; dz -= N * np.round(dz / N)
        best = np.minimum(best, np.sqrt(dx * dx + dy * dy + dz * dz) - R)
    return best, R

def drag(SolverCls, N, phi=PHI0, centres=CENTRE, mu=0.1, F=1e-3, dt=80.0,
         warm_tol=1e-6, tail=40, max_steps=1000):
    sdf, R = sphere_sdf(N, phi, centres)
    s = SolverCls(N, N, N)
    s.set_rho(1.0); s.set_mu(mu); s.set_dt(dt)
    s.set_body_force(F, 0, 0); s.set_advection(False)     # Stokes: inertia off
    s.set_velocity_solver_params(150)
    s.set_pressure_multigrid(True, levels=max(2, int(np.log2(N)) - 1))
    s.set_pressure_pcg(True, 200, 1e-8)
    s.set_solid(sdf, cutcell_pressure=True, pressure_coarse="rediscretized")

    prev, warm, um, pit, t0 = 0.0, None, [], [], time.time()
    for it in range(max_steps):
        s.step()
        um.append(float(s.get_u().mean()))            # rolling history of <u>
        pit.append(int(s.last_pressure_iterations()))
        if warm is None:                              # still relaxing to steady state
            if it % 10 == 9:
                if it > 10 and abs(um[-1] - prev) < warm_tol * (abs(um[-1]) + 1e-30):
                    warm = it                         # reached moderate steady state
                prev = um[-1]
        elif it - warm >= tail:                       # collected a full post-warm tail
            break

    umean = float(np.mean(um[-tail:]))                # tail-average damps the jitter
    K = F * N ** 3 / len(centres) / (6 * np.pi * mu * R * umean)
    return dict(N=N, K=K, k=mu * umean / F, steps=it + 1, warm=warm,
                pit=float(np.mean(pit[-tail:])), secs=time.time() - t0)

## Sweep both grids

In [ ]:
#| label: sweep
Ns = [16, 24, 32, 40]
grids = {"staggered": flow.Solver, "collocated": flow.SolverColocated}
data = {name: [drag(cls, N) for N in Ns] for name, cls in grids.items()}

for name, rows in data.items():
    print(f"{name:>11s}: " + "  ".join(
        f"N={r['N']}→K={r['K']:.4f}({100*(r['K']-K_ZH)/K_ZH:+.2f}%)" for r in rows))

## Order and limit of each grid

To separate *"is it converging"* from *"does it converge to the right answer"*, we
fit each grid's drag to

$$
K(N)=K_\infty + C\,N^{-p},
$$ {#eq-fit}

where $K_\infty$ is the grid's **own** continuum limit and $p$ its observed order.
The fit is linear in $(K_\infty, C)$ for any fixed $p$, so a one-dimensional scan
over $p$ (least squares at each step) finds all three with nothing but NumPy. When
the error is *bias-dominated* — the values hover near their limit within scatter
rather than decaying monotonically — there is no clean power law, and the fit
degenerates to "flat at $K_\infty$" (huge $p$, tiny $C$). We flag that case instead
of reporting a meaningless order.

In [ ]:
#| label: fit
def fit_order(Ns, Ks, ps=np.linspace(0.5, 4.0, 701)):
    Ns = np.asarray(Ns, float); Ks = np.asarray(Ks, float)
    best = None
    for p in ps:
        A = np.column_stack([np.ones_like(Ns), Ns ** (-p)])   # columns: K_inf, C
        coef, *_ = np.linalg.lstsq(A, Ks, rcond=None)
        r = float(np.sum((A @ coef - Ks) ** 2))
        if best is None or r < best[0]:
            best = (r, p, coef[0], coef[1])
    _, p, Kinf, C = best
    monotone = np.all(np.diff(Ks) > 0) or np.all(np.diff(Ks) < 0)
    clean = monotone and p < 3.5           # a genuine, resolvable power law
    return dict(p=p, Kinf=Kinf, C=C, clean=clean)

fits = {name: fit_order(Ns, [r["K"] for r in rows]) for name, rows in data.items()}
for name, f in fits.items():
    bias = 100 * (f["Kinf"] - K_ZH) / K_ZH
    order = f"order p≈{f['p']:.2f}" if f["clean"] else "no clean power law (bias-dominated)"
    print(f"{name:>11s}: {order:<38s} K∞={f['Kinf']:.4f}  bias vs Z&H {bias:+.2f}%")

## The result

In [ ]:
#| label: fig-convergence
#| fig-cap: "Left: drag factor vs resolution. The staggered grid converges onto the Zick & Homsy value (dashed); the collocated grid hovers ~1% high. Centre: signed relative error — staggered heads to zero, collocated to a non-zero floor (the velocity-placement bias, not a transient). Right: wall-clock to steady state (single 6-thread CPU; indicative) — the collocated approximate projection is the costlier of the two."
fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(12.5, 3.9))
col = {"staggered": BLUE, "collocated": RED}
Nf = np.linspace(min(Ns) * 0.95, max(Ns) * 1.05, 100)

a0.axhline(K_ZH, ls="--", color="0.3", lw=1, label=f"Z&H  K={K_ZH:.3f}")
for name, rows in data.items():
    K = [r["K"] for r in rows]; f = fits[name]
    a0.plot(Ns, K, "o", color=col[name], label=f"{name}  (K∞={f['Kinf']:.3f})")
    a0.plot(Nf, f["Kinf"] + f["C"] * Nf ** (-f["p"]), "-", color=col[name], lw=1, alpha=.7)
a0.set(xlabel="N (cells per period)", ylabel="drag factor K", title="convergence")
a0.legend(fontsize=8)

for name, rows in data.items():
    err = 100 * (np.array([r["K"] for r in rows]) - K_ZH) / K_ZH
    a1.plot(Ns, err, "o-", color=col[name], label=name)
a1.axhline(0, color="0.3", ls="--", lw=1)
a1.set(xlabel="N", ylabel="(K − K$_{ZH}$) / K$_{ZH}$  [%]", title="signed error → limit")
a1.legend(fontsize=8)

for name, rows in data.items():
    a2.plot(Ns, [r["secs"] for r in rows], "o-", color=col[name], label=name)
a2.set(xlabel="N", ylabel="wall-clock to steady [s]", title="solve cost")
a2.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
#| label: tbl
print(f"{'grid':>11s} {'N':>4s} {'K':>8s} {'err%':>7s} {'p_iter':>7s} {'steps':>6s} {'sec':>6s}")
for name, rows in data.items():
    for r in rows:
        print(f"{name:>11s} {r['N']:>4d} {r['K']:>8.4f} "
              f"{100*(r['K']-K_ZH)/K_ZH:>+6.2f} {r['pit']:>7.0f} {r['steps']:>6d} {r['secs']:>6.1f}")

## What it means

- **The staggered grid converges — cleanly, and onto the benchmark.** Its drag rises
  monotonically with resolution and fits $K_\infty + C\,N^{-p}$ with $p\approx2$; the
  extrapolated limit $K_\infty$ lands on the Zick & Homsy value to a fraction of a
  percent. This is the cut-cell IBM resolving the curved sphere surface at the
  expected second order — the accurate grid for drag and permeability.
- **The collocated grid does *not* converge onto it.** Its error is **bias-dominated**:
  the drag sits ~1% above Z&H across the whole sweep and only jitters at the
  sub-percent level with refinement — there is no clean decaying power law, because
  the dominant error is a *resolution-independent* offset, not a discretization term
  that shrinks with $h$. Refining the mesh does not remove it. It is an intrinsic
  property of the cell-centred velocity placement plus its ABC *approximate*
  projection — not a transient and not a bug. (This is exactly why the staggered grid
  is *the* flow solver in the suite; the collocated variant exists for coupling and
  single-grid convenience, with this accuracy trade understood.)
- **Cost.** The collocated approximate projection also leaves the mean velocity
  jittering — it never reaches the machine-tight steady state the staggered grid's
  *exact* projection does, so it needs more steps to a fair tolerance (hence the
  tail-averaging). On this problem the staggered grid is both more accurate *and*
  cheaper.

The headline for a practitioner: **use the staggered `flow.Solver` for
permeability/drag**; the ~1% collocated offset is a property of *where the velocities
live*, measurable here and irreducible by mesh refinement.

## Adapt this yourself

- **Change the solid fraction.** Raise `PHI0` toward close packing ($\phi_\max=\pi/6$
  for simple cubic) and update `K_ZH` from the Zick & Homsy table (see the
  [Zick–Homsy example](../../examples/zick-homsy/index.qmd)); the bias trend persists.
- **Add finer grids.** Append `48, 64` to `Ns` (GPU-territory on wall-clock) to pin
  down each $K_\infty$ more tightly and sharpen the fitted order.
- **Swap the quantity of interest.** The same head-to-head applies to any QoI with a
  ground truth — a Poiseuille profile, a Taylor–Green decay rate — to map *where* the
  two placements agree and where the collocated bias shows up.

## Reproduce this

```bash
pip install peclet
quarto render benchmarks/staggered-vs-collocated/index.qmd --execute
# ...or against a local source build:
PECLET_LOCAL_BUILD=/path/to/suite/flow/build_mpi OMP_NUM_THREADS=6 \
  OMP_PROC_BIND=spread OMP_PLACES=threads \
  quarto render benchmarks/staggered-vs-collocated/index.qmd --execute
```

::: {.callout-note}
## Ground truth
Zick, A. A. & Homsy, G. M. (1982), *Stokes flow through periodic arrays of spheres*,
J. Fluid Mech. **115**, 13–26 — the semi-analytic drag factors this page converges
against.
:::